**Notebook 00:**
# Descarga del conjunto de datos CIC-IDS2017 desde Kaggle

Este notebook descarga de forma automatizada el conjunto de datos CIC-IDS2017 desde Kaggle, utilizando la API oficial. Se incluye la configuracion de autenticacion, la descarga, la verificacion de integridad y el registro en log.

Dataset en Kaggle: https://www.kaggle.com/datasets/chethuhn/network-intrusion-dataset

---

## 1. Requisitos

```bash
pip install kaggle pandas
```

Espacio en disco requerido: ~3 GB.

## 2. Configuracion del API Key de Kaggle

Para descargar datasets desde Kaggle mediante la API, es necesario autenticarse con una cuenta de usuario. El proceso es el siguiente:

### 2.1. Obtencion del archivo `kaggle.json`

1. Crear una cuenta en https://www.kaggle.com/ 
2. Iniciar sesion y acceder a **Settings** desde el menu de perfil.
3. En la seccion **API**, hacer clic en **Create New Token**.
4. Se descargara automaticamente un archivo llamado `kaggle.json` con el siguiente contenido:

```json
{"username": "tu_usuario", "key": "tu_api_key"}
```

### 2.2. Ubicacion del archivo

El archivo `kaggle.json` debe colocarse en el directorio `.kaggle` dentro de la carpeta de usuario del sistema:

Windows: `C:\Users\<Usuario>\.kaggle\kaggle.json`

Si la carpeta `.kaggle` no existe, debe crearse manualmente.

In [41]:
import os
import hashlib
from pathlib import Path
from datetime import datetime

import pandas as pd
import kaggle

print(f"Fecha de ejecucion: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Directorio de trabajo: {os.getcwd()}")

Fecha de ejecucion: 2026-10-11 22:06:57
Directorio de trabajo: c:\Proyectos\TT-deteccionIntrusiones


In [25]:
# ============================================================
# Autenticacion con la API de Kaggle
# ============================================================

try:
    kaggle.api.authenticate()
    print("Autenticacion con Kaggle exitosa.")
except Exception as e:
    print(f"Error de autenticacion: {e}")
    raise SystemExit("No se puede continuar sin autenticacion.")

Autenticacion con Kaggle exitosa.


In [42]:
# ============================================================
# Configuracion de rutas
# ============================================================

PROJECT_ROOT = Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
LOG_DIR = PROJECT_ROOT / "logs"
LOG_FILE = LOG_DIR / "download_log.csv"

for d in [RAW_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DATASET_SLUG = "chethuhn/network-intrusion-dataset"

print(f"Directorio de datos: {RAW_DIR}")
print(f"Directorio de logs: {LOG_DIR}")
print(f"Dataset de Kaggle: {DATASET_SLUG}")

Directorio de datos: C:\Proyectos\TT-deteccionIntrusiones\data\raw
Directorio de logs: C:\Proyectos\TT-deteccionIntrusiones\logs
Dataset de Kaggle: chethuhn/network-intrusion-dataset


In [43]:
# ============================================================
# Archivos CSV esperados del dataset CIC-IDS2017
# ============================================================

EXPECTED_FILES = [
    "Monday-WorkingHours.pcap_ISCX.csv",
    "Tuesday-WorkingHours.pcap_ISCX.csv",
    "Wednesday-workingHours.pcap_ISCX.csv",
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
]

print(f"Archivos esperados: {len(EXPECTED_FILES)}")
for f in EXPECTED_FILES:
    print(f"  - {f}")

Archivos esperados: 8
  - Monday-WorkingHours.pcap_ISCX.csv
  - Tuesday-WorkingHours.pcap_ISCX.csv
  - Wednesday-workingHours.pcap_ISCX.csv
  - Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
  - Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  - Friday-WorkingHours-Morning.pcap_ISCX.csv
  - Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
  - Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv


In [44]:
# ============================================================
# Funcion auxiliar
# ============================================================

def compute_sha256(filepath: Path) -> str:
    """
    Calcula el hash SHA-256 de un archivo.
    """
    sha256 = hashlib.sha256()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            sha256.update(chunk)
    return sha256.hexdigest()

In [45]:
# ============================================================
# Descarga del dataset desde Kaggle
# ============================================================

already_complete = all(
    (RAW_DIR / f).exists() and (RAW_DIR / f).stat().st_size > 0
    for f in EXPECTED_FILES
)

if already_complete:
    print("Todos los archivos ya existen en data/raw/. Se omite la descarga.")
else:
    print("Descargando dataset desde Kaggle...")
    print(f"Dataset: {DATASET_SLUG}")
    print(f"Destino: {RAW_DIR}")

    try:
        kaggle.api.dataset_download_files(
            DATASET_SLUG,
            path=str(RAW_DIR),
            unzip=True,
        )
        print("\nDescarga y descompresion completadas.")
    except Exception as e:
        print(f"\nError durante la descarga: {e}")
        raise

Descargando dataset desde Kaggle...
Dataset: chethuhn/network-intrusion-dataset
Destino: C:\Proyectos\TT-deteccionIntrusiones\data\raw
Dataset URL: https://www.kaggle.com/datasets/chethuhn/network-intrusion-dataset

Descarga y descompresion completadas.


In [52]:
# ============================================================
# Verificacion de integridad y registro en log
# ============================================================

results = []
all_valid = True
timestamp = datetime.now().isoformat()

print("Verificacion de integridad\n")

for fname in EXPECTED_FILES:
    path = RAW_DIR / fname

    row = {
        "timestamp": timestamp,
        "source": "kaggle",
        "file": fname,
        "status": "",
        "size_bytes": 0,
        "size_mb": 0.0,
        "sha256": "",
        "n_columns": None,
        "error": ""
    }

    if not path.exists():
        row["status"] = "missing"
        row["error"] = "Archivo ausente"
        all_valid = False
        print(f"[ERROR] {fname}: ausente")
        results.append(row)
        continue

    size_bytes = path.stat().st_size
    row["size_bytes"] = size_bytes
    row["size_mb"] = round(size_bytes / (1024 ** 2), 2)

    if size_bytes == 0:
        row["status"] = "empty"
        row["error"] = "Archivo vacio"
        all_valid = False
        print(f"[ERROR] {fname}: vacio")
        results.append(row)
        continue

    try:
        header = pd.read_csv(path, nrows=1, low_memory=False)
        n_columns = len(header.columns)
        sha256 = compute_sha256(path)

        row["status"] = "valid"
        row["n_columns"] = n_columns
        row["sha256"] = sha256

        print(f"[OK] {fname} | {row['size_mb']:.2f} MB | {n_columns} columnas")

    except Exception as e:
        row["status"] = "error"
        row["error"] = str(e)
        all_valid = False
        print(f"[ERROR] {fname}: {e}")

    results.append(row)

# Guardar registro en log CSV
df_log = pd.DataFrame(results)

LOG_DIR.mkdir(parents=True, exist_ok=True)
write_header = not LOG_FILE.exists()
df_log.to_csv(LOG_FILE, mode="a", header=write_header, index=False)

n_valid = sum(1 for r in results if r["status"] == "valid")
total_mb = sum(r["size_mb"] for r in results)

print()
print(f"Archivos validos: {n_valid}/{len(EXPECTED_FILES)}")
print(f"Tamano total: {total_mb:.2f} MB")
print(f"Log guardado en: {LOG_FILE}")

if all_valid:
    print("Todos los archivos esperados fueron verificados correctamente.")
else:
    print("Se detectaron archivos ausentes, vacios o con error.")

Verificacion de integridad

[OK] Monday-WorkingHours.pcap_ISCX.csv | 168.73 MB | 79 columnas
[OK] Tuesday-WorkingHours.pcap_ISCX.csv | 128.82 MB | 79 columnas
[OK] Wednesday-workingHours.pcap_ISCX.csv | 214.74 MB | 79 columnas
[OK] Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv | 49.61 MB | 79 columnas
[OK] Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv | 79.25 MB | 79 columnas
[OK] Friday-WorkingHours-Morning.pcap_ISCX.csv | 55.62 MB | 79 columnas
[OK] Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv | 73.34 MB | 79 columnas
[OK] Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv | 73.55 MB | 79 columnas

Archivos validos: 8/8
Tamano total: 843.66 MB
Log guardado en: C:\Proyectos\TT-deteccionIntrusiones\logs\download_log.csv
Todos los archivos esperados fueron verificados correctamente.


In [ ]:
# ============================================================
# Verificacion de integridad contra el log de descarga
# ============================================================

if not LOG_FILE.exists():
    raise FileNotFoundError(f"No existe el archivo de log: {LOG_FILE}")

df_log = pd.read_csv(LOG_FILE)

df_latest = (
    df_log
    .sort_values("timestamp")
    .groupby("file", as_index=False)
    .last()
)

print("Verificacion de integridad contra el log\n")

all_verified = True

for _, row in df_latest.iterrows():
    fname = row["file"]
    path = RAW_DIR / fname
    expected_sha256 = row["sha256"]

    if not path.exists():
        print(f"[FALTA] {fname}")
        all_verified = False
        continue

    current_sha256 = compute_sha256(path)

    if pd.isna(expected_sha256) or str(expected_sha256).strip() == "":
        print(f"[SIN HASH] {fname}: no hay hash registrado en el log")
        all_verified = False
        continue

    if current_sha256 == expected_sha256:
        print(f"[OK] {fname}")
    else:
        print(f"[MODIFICADO] {fname}")
        print(f"   Hash esperado: {expected_sha256}")
        print(f"   Hash actual:   {current_sha256}")
        all_verified = False

print()

if all_verified:
    print("Todos los archivos coinciden con el hash registrado en el log.")
else:
    print("Se detectaron discrepancias. Revisar los archivos antes de continuar.")

Verificacion de integridad contra el log

[OK] Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
[OK] Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
[OK] Friday-WorkingHours-Morning.pcap_ISCX.csv
[OK] Monday-WorkingHours.pcap_ISCX.csv
[OK] Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
[OK] Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
[OK] Tuesday-WorkingHours.pcap_ISCX.csv
[OK] Wednesday-workingHours.pcap_ISCX.csv

Todos los archivos coinciden con el hash registrado en el log.


## 3. Nota sobre la fuente de datos

El dataset se obtuvo desde Kaggle como fuente alternativa al servidor oficial del Canadian Institute for Cybersecurity (UNB), debido a la intermitencia del servidor institucional. El contenido de los archivos corresponde al mismo conjunto CIC-IDS2017 descrito en:

> I. Sharafaldin, A. H. Lashkari, and A. A. Ghorbani, "Toward Generating a New Intrusion Detection Dataset and Intrusion Traffic Characterization", 4th International Conference on Information Systems Security and Privacy (ICISSP), Portugal, January 2018.

La referencia oficial del dataset permanece como: https://www.unb.ca/cic/datasets/ids-2017.html

El registro de descarga con hash SHA-256 se conserva en `logs/download_log.csv` para garantizar la trazabilidad y reproducibilidad del experimento.